In [2]:
# =============================================================
# CELL 1: CONFIGURATION (Adaptado para PDF)
# =============================================================

import os

# --- Paths ---
PROJECT_DIR = '.' # Caminho relativo seguro!
ENVISION_INDEX_PATH = os.path.join(PROJECT_DIR, 'envision_index.json')
SYSTEM_PROMPT_PATH = os.path.join(PROJECT_DIR, 'rag_system_prompt.txt')

# Lista com os seus arquivos .pdf (altere o nome se necessário)
PDF_FILES = [ 
    os.path.join(PROJECT_DIR, 'sample_project.pdf'),
]

# --- Model settings ---
MODEL_NAME = 'llama3.3:70b'
OLLAMA_URL = 'http://localhost:11434'
TEMPERATURE = 0.1 
CONTEXT_WINDOW = 128000 
MAX_OUTPUT_TOKENS = 8192 
NUM_RUNS = 3 # Avaliando 3 vezes para confiabilidade

# --- RAG settings ---
EMBEDDING_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
CHROMA_DB_PATH = os.path.join(PROJECT_DIR, 'chroma_db')
RETRIEVAL_TOP_K = 15

# --- Thread limits (segurança do servidor) ---
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['MKL_NUM_THREADS'] = '4'
os.environ['OPENBLAS_NUM_THREADS'] = '4'
os.environ['NUMEXPR_MAX_THREADS'] = '4'

# --- Results directory ---
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

print('✅ Configuração carregada para experimento PDF.')
print(f'   Model: {MODEL_NAME}')
print(f'   PDF files: {len(PDF_FILES)}')

✅ Configuração carregada para experimento PDF.
   Model: llama3.3:70b
   PDF files: 1


In [3]:
# =============================================================
# CELL 2: HELPER FUNCTIONS (Adaptado para PDF)
# =============================================================

import json, time, datetime, traceback, hashlib
import requests
import pandas as pd
import fitz  # PyMuPDF para ler PDFs

# ---------- PDF EXTRACTION ----------
def extract_pdf_metadata(filepath):
    """Extrai todo o texto de um arquivo PDF usando PyMuPDF."""
    text_content = ""
    try:
        doc = fitz.open(filepath)
        for page in doc:
            text_content += page.get_text("text") + "\n"
        doc.close()
    except Exception as e:
        print(f"Erro ao ler o PDF {filepath}: {e}")
        
    return {
        'source_file': os.path.basename(filepath),
        'extracted_text': text_content
    }

# ---------- OLLAMA API ----------
def call_ollama(system_prompt, user_message, timeout=7200): # TIMEOUT SEGURO!
    """Envia o prompt para a API do Ollama e retorna a resposta."""
    start = time.time()
    try:
        r = requests.post(
            f'{OLLAMA_URL}/api/generate',
            json={
                'model': MODEL_NAME,
                'system': system_prompt,
                'prompt': user_message,
                'stream': False,
                'options': {
                    'num_ctx': CONTEXT_WINDOW,
                    'temperature': TEMPERATURE,
                    'num_predict': MAX_OUTPUT_TOKENS,
                }
            },
            timeout=timeout 
        )
        elapsed = time.time() - start
        result = r.json()
        return {
            'response': result.get('response', ''),
            'elapsed_seconds': round(elapsed, 2),
            'eval_count': result.get('eval_count', 0),
            'prompt_eval_count': result.get('prompt_eval_count', 0),
            'success': True
        }
    except Exception as e:
        elapsed = time.time() - start
        return {
            'response': f'ERROR: {str(e)}',
            'elapsed_seconds': round(elapsed, 2),
            'eval_count': 0,
            'prompt_eval_count': 0,
            'success': False
        }

# ---------- ENVISION DATA LOADER ----------
def load_envision_by_category():
    """Carrega os dados JSON do Envision separados por categoria."""
    with open(ENVISION_INDEX_PATH) as f:
        data = json.load(f)
    categories = {}
    for cid, credit in data['credits'].items():
        cat = credit['sheet']
        if cat not in categories:
            categories[cat] = {}
        categories[cat][cid] = credit
    return categories, data['metadata']

# ---------- RESULT SAVING ----------
def save_result(result_dict, filepath):
    """Salva os dicionarios como arquivos JSON."""
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, 'w') as f:
        json.dump(result_dict, f, indent=2, default=str)
    print(f'   Salvo: {filepath}')

def timestamp():
    return datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

print('✅ Funções auxiliares (PyMuPDF & Ollama) carregadas com sucesso.')

✅ Funções auxiliares (PyMuPDF & Ollama) carregadas com sucesso.


In [4]:
# =============================================================
# CELL 3: BUILD RAG INDEX 
# =============================================================

import chromadb
import llama_index.core
from llama_index.core import Document, VectorStoreIndex, StorageContext
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.ollama import Ollama
from llama_index.core import Settings

# TIMEOUT SEGURO NO LLAMAINDEX!
Settings.llm = Ollama(
    model=MODEL_NAME, 
    request_timeout=7200, 
    temperature=TEMPERATURE
)
Settings.embed_model = HuggingFaceEmbedding(
    model_name=EMBEDDING_MODEL
)

print('Construindo o índice vetorial a partir do envision_index.json...')

with open(ENVISION_INDEX_PATH) as f:
    envision_raw = json.load(f)

documents = []
for credit_id, credit in envision_raw['credits'].items():
    text = f"""Credit: {credit['credit_id']} - {credit['credit_name']}
Category: {credit['sheet']} / {credit['category']}
Points: {credit['points_display']}
{credit['intent']}
{credit['metric']}
{credit['applicability_description']}
Questions:"""
    for q in credit.get('questions', []):
        text += f"\n  {q['letter']}: {q['text']}"
        
    if credit.get('lookup_criteria'):
        text += '\n\nLookup Criteria:'
        for lc in credit['lookup_criteria']:
            text += f"\n  {lc['criterion']}: {lc['description']}"
            
    tab = credit.get('tabulation', {})
    if tab:
        text += '\n\nLevel Guides:'
        for lvl in ['Improved', 'Enhanced', 'Superior', 'Conserving']:
            g = tab.get(f'LevelGuide_{lvl}', '')
            if g:
                text += f"\n  {lvl}: {g}"
                
    pts = credit.get('points', {})
    if pts:
        text += f"""\n\nPoints Rubric:
No Level={pts.get('No_Level',0)}, 
Improved={pts.get('Improved',0)}, 
Enhanced={pts.get('Enhanced',0)}, 
Superior={pts.get('Superior',0)}, 
Conserving={pts.get('Conserving',0)}, 
Restorative={pts.get('Restorative','N/A')}"""
        
    doc = Document(
        text=text,
        metadata={
            'credit_id': credit_id,
            'credit_name': credit['credit_name'],
            'category': credit['sheet'],
            'subcategory': credit['category'],
            'max_points': pts.get('Total', 0),
            'has_lookup': bool(credit.get('lookup_criteria')),
        }
    )
    documents.append(doc)

db_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

try:
    db_client.delete_collection('envision_credits')
except:
    pass

chroma_collection = db_client.create_collection('envision_credits')
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

rag_index = VectorStoreIndex.from_documents(
    documents, 
    storage_context=storage_context, 
    embed_model=Settings.embed_model,
    show_progress=True
)

print(f'\n✅ RAG index construído e salvo com sucesso em {CHROMA_DB_PATH}')

/export/livia/home/vision/Gbaldessin/miniconda3/lib/python3.13/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Construindo o índice vetorial a partir do envision_index.json...


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/59 [00:00<?, ?it/s]


✅ RAG index construído e salvo com sucesso em ./chroma_db


In [6]:
# =============================================================
# CELL 4: WEEKEND RUNNER (Adaptado para PDF e Timeouts)
# =============================================================

print('=' * 70)
print(' ENVISION PDF EXPERIMENT RUNNER')
print(f' Started:  {datetime.datetime.now()}')
print(f' Model:    {MODEL_NAME}')
print(f' Files:    {len(PDF_FILES)} PDF file(s)')
print(f' Runs:     {NUM_RUNS} per experiment')
print('=' * 70)

# Carrega o prompt base
with open(SYSTEM_PROMPT_PATH) as f:
    system_prompt = f.read()

# Carrega as categorias do Envision
categories, meta = load_envision_by_category()
category_names = list(categories.keys())

# Engine do RAG
rag_query_engine = rag_index.as_query_engine(
    similarity_top_k=RETRIEVAL_TOP_K,
    system_prompt=system_prompt
)

timing_log = []

for file_idx, pdf_path in enumerate(PDF_FILES):
    file_label = os.path.splitext(os.path.basename(pdf_path))[0]
    print(f'\n{"=" * 70}')
    print(f'FILE {file_idx+1}/{len(PDF_FILES)}: {file_label}')
    print(f'{"=" * 70}')

    # --- Extração do PDF ---
    print('\n Extraindo texto do PDF...')
    try:
        pdf_data = extract_pdf_metadata(pdf_path)
        pdf_json = json.dumps(pdf_data, indent=2, default=str)
        print(f' ✅ Texto extraído do PDF com sucesso.')
    except Exception as e:
        print(f' ❌ Falha na extração do PDF: {e}')
        print(f' Pulando este arquivo.')
        continue

    # Salva o texto extraído para referência futura
    save_result(pdf_data, os.path.join(RESULTS_DIR, file_label, 'pdf_metadata.json'))

    # =====================================================
    # EXPERIMENT A: ZERO-SHOT
    # =====================================================
    for run_num in range(1, NUM_RUNS + 1):
        run_dir = os.path.join(RESULTS_DIR, file_label, 'zero_shot', f'run_{run_num}')
        os.makedirs(run_dir, exist_ok=True)
        print(f'\n --- ZERO-SHOT | Run {run_num}/{NUM_RUNS} ---')
        
        run_start = time.time()
        all_category_results = {}
        
        for cat_name, cat_credits in categories.items():
            print(f'   Category: {cat_name} ({len(cat_credits)} credits)...', end=' ')
            
            cat_json = json.dumps(cat_credits, indent=2, default=str)
            user_msg = f"""## Envision Reference Data — {cat_name} Category
{cat_json}

## Project PDF Metadata
{pdf_json}

Task: Evaluate this project against all {cat_name} credits shown above. For each credit, determine applicability, evaluate questions against the PDF metadata, determine the achievable level, calculate points, and identify documentation gaps. Return your assessment as JSON following the credit_assessments format in your system instructions."""
            
            # TIMEOUT DE 1 HORA (3600s) ATIVO!
            result = call_ollama(system_prompt, user_msg, timeout=7200) 
            
            cat_safe = cat_name.replace(' ', '_').lower()
            save_result(result, os.path.join(run_dir, f'{cat_safe}.json'))
            all_category_results[cat_name] = result
            
            status = '✅' if result['success'] else '❌'
            print(f'{status} ({result["elapsed_seconds"]}s)')
            
        run_elapsed = round(time.time() - run_start, 2)
        summary = {
            'experiment': 'zero_shot', 'file': file_label, 'run': run_num,
            'model': MODEL_NAME, 'temperature': TEMPERATURE,
            'total_elapsed_seconds': run_elapsed,
            'categories': {
                k: {
                    'elapsed_seconds': v['elapsed_seconds'],
                    'output_tokens': v['eval_count'],
                    'input_tokens': v['prompt_eval_count'],
                    'success': v['success'],
                } for k, v in all_category_results.items()
            },
            'timestamp': timestamp()
        }
        save_result(summary, os.path.join(run_dir, '_summary.json'))
        timing_log.append({'experiment': 'zero_shot', 'file': file_label, 'run': run_num, 'seconds': run_elapsed})

    # =====================================================
    # EXPERIMENT B: RAG
    # =====================================================
    for run_num in range(1, NUM_RUNS + 1):
        run_dir = os.path.join(RESULTS_DIR, file_label, 'rag', f'run_{run_num}')
        os.makedirs(run_dir, exist_ok=True)
        print(f'\n --- RAG | Run {run_num}/{NUM_RUNS} ---')
        
        run_start = time.time()
        all_category_results = {}
        
        for cat_name in category_names:
            n_credits = len(categories[cat_name])
            print(f'   Category: {cat_name} ({n_credits} credits)...', end=' ')
            
            query_text = f"""Evaluate this infrastructure project against all Envision credits in the {cat_name} category. 
Project PDF Metadata: {pdf_json} 
For each credit in {cat_name}, determine applicability, evaluate questions, determine the achievable level, calculate points, and identify documentation gaps. Return JSON following the credit_assessments format."""
            
            rag_start = time.time()
            try:
                rag_response = rag_query_engine.query(query_text)
                rag_elapsed = round(time.time() - rag_start, 2)
                
                sources = []
                if hasattr(rag_response, 'source_nodes'):
                    for node in rag_response.source_nodes:
                        sources.append({
                            'credit_id': node.metadata.get('credit_id', '?'),
                            'score': round(node.score, 4) if node.score else None,
                            'category': node.metadata.get('category', '?'),
                        })
                        
                result = {
                    'response': str(rag_response),
                    'elapsed_seconds': rag_elapsed,
                    'retrieved_sources': sources,
                    'num_sources': len(sources),
                    'success': True
                }
                status = '✅'
                
            except Exception as e:
                rag_elapsed = round(time.time() - rag_start, 2)
                result = {
                    'response': f'ERROR: {str(e)}',
                    'elapsed_seconds': rag_elapsed,
                    'retrieved_sources': [],
                    'num_sources': 0,
                    'success': False
                }
                status = '❌'
                
            cat_safe = cat_name.replace(' ', '_').lower()
            save_result(result, os.path.join(run_dir, f'{cat_safe}.json'))
            all_category_results[cat_name] = result
            
            print(f'{status} ({result["elapsed_seconds"]}s, {result["num_sources"]} sources)')
            
        run_elapsed = round(time.time() - run_start, 2)
        summary = {
            'experiment': 'rag', 'file': file_label, 'run': run_num,
            'model': MODEL_NAME, 'temperature': TEMPERATURE,
            'retrieval_top_k': RETRIEVAL_TOP_K, 'embedding_model': EMBEDDING_MODEL,
            'total_elapsed_seconds': run_elapsed,
            'categories': {
                k: {
                    'elapsed_seconds': v['elapsed_seconds'],
                    'num_sources': v['num_sources'],
                    'success': v['success'],
                } for k, v in all_category_results.items()
            },
            'timestamp': timestamp()
        }
        save_result(summary, os.path.join(run_dir, '_summary.json'))
        timing_log.append({'experiment': 'rag', 'file': file_label, 'run': run_num, 'seconds': run_elapsed})

# =========================================================
# FINAL SUMMARY
# =========================================================
print('\n' + '=' * 70)
print(' ALL EXPERIMENTS COMPLETE')
print(f' Finished: {datetime.datetime.now()}')
print('=' * 70)

timing_df = pd.DataFrame(timing_log)
timing_df.to_csv(os.path.join(RESULTS_DIR, 'timing_summary.csv'), index=False)
print('\nTiming summary:')
print(timing_df.to_string(index=False))

 ENVISION PDF EXPERIMENT RUNNER
 Started:  2026-04-22 21:44:47.703440
 Model:    llama3.3:70b
 Files:    1 PDF file(s)
 Runs:     3 per experiment

FILE 1/1: sample_project

 Extraindo texto do PDF...
 ✅ Texto extraído do PDF com sucesso.
   Salvo: ./results/sample_project/pdf_metadata.json

 --- ZERO-SHOT | Run 1/3 ---
   Category: Climate And Resilience (9 credits)... 

OSError: [Errno 28] No space left on device: './results/sample_project/zero_shot/run_1/climate_and_resilience.json'